# Part A — Visualization tasks A4 and A7

This notebook covers two visualization tasks from Part A of the Network Analysis
assignment:

* **A4** — Visualize the selected movie network (*2008 — The Dark Knight*) the way
  you would in **Gephi** and **Cytoscape**. Because this machine has no graphical
  display (it is a headless cluster node), we do two things: (1) we **export** the
  network into the exact file formats that Gephi and Cytoscape import, so you can
  open them in the real programs and take the official screenshots, and (2) we
  draw **emulated renders** with matplotlib that mimic what those programs would
  show, so the intended result is visible even without the GUI.
* **A7** — Build and visualize a **Lord of the Rings "couples" network**: the set
  of canonical romantic pairings from Tolkien's legendarium. Each character gets a
  **gender** and a **race**. We then draw the network with **vertex colour = gender**
  and **vertex shape = race**, again exporting Gephi/Cytoscape files plus an
  emulated matplotlib render.

**A note on vocabulary, in plain words.** A *network* (also called a *graph*) is
just a set of dots, called *nodes* or *vertices*, joined by lines, called *edges*.
In the Dark Knight network each dot is a character and a line means "these two
characters appeared together on screen"; the line carries a *weight*, which is
simply how many times they co-appeared. A *layout* is the recipe a program uses to
decide where to place each dot on the page so the picture is readable.


## Setup

We load the shared helper module `na_utils` (it lives in `src/`), force matplotlib
into its non-interactive **Agg** backend (the backend that draws straight to an
image file instead of to a screen — essential on a headless machine), and fix the
random seeds. Layout algorithms start from random positions, so fixing the seed
makes the pictures reproducible: re-running the notebook gives the same drawing.

In [1]:
import sys, os, json, random
sys.path.insert(0, "/home/mickaelz/Network analysis/src")
import na_utils as na

import numpy as np
import networkx as nx
import matplotlib
na.set_style()                 # forces the Agg backend + nice defaults
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D   # used to build custom legends

# Reproducible layouts: fix every random source we might touch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/home/mickaelz/Network analysis"
EXPORT_DIR = os.path.join(BASE, "exports")
FIG_DIR = os.path.join(BASE, "figures")
os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("networkx", nx.__version__)
print("matplotlib", matplotlib.__version__, "backend:", matplotlib.get_backend())


networkx 3.2.1
matplotlib 3.9.4 backend: Agg


# Task A4 — Cytoscape + Gephi visualization of the selected network

The selected network is **`2008_The_Dark_Knight`** (the constant
`na.SELECTED_MOVIE`). It is an **undirected, weighted** *character co-appearance*
graph: each node is a character, each edge means the two characters appeared in the
same scene, and the edge `weight` counts how many co-appearances there were.

The assignment asks for a visualization where **vertex size grows with vertex
degree**. The *degree* of a node is the number of edges touching it — in plain
terms, how many different characters that character ever shared a scene with. A
character who interacts with many others (like Batman) has a high degree and should
be drawn as a big dot.

In [2]:
# Load the selected movie network.
G = na.load_movie_graph(na.SELECTED_MOVIE)
print("Movie:", na.SELECTED_MOVIE)
print("Nodes (characters):", G.number_of_nodes())
print("Edges (co-appearances):", G.number_of_edges())
print("Directed?", G.is_directed(), "| Weighted? edge has 'weight':",
      "weight" in next(iter(G.edges(data=True)))[2])

# Peek at one node and one edge so the structure is concrete.
sample_node = list(G.nodes(data=True))[0]
sample_edge = list(G.edges(data=True))[0]
print("\nExample node:", sample_node)
print("Example edge:", sample_edge)


Movie: 2008_The_Dark_Knight
Nodes (characters): 25
Edges (co-appearances): 106
Directed? False | Weighted? edge has 'weight': True

Example node: ('Bruce Wayne / Batman', {'first': 16, 'last': 3550, 'role': 'Christian Bale'})
Example edge: ('Bruce Wayne / Batman', 'Lucius Fox', {'first': 16, 'last': 3559, 'weight': 44})


### Step 1 — Add `degree` and `weighted_degree` node attributes

Before exporting, we attach two numbers to every node so that inside Gephi or
Cytoscape you can map **node size → degree** with a couple of clicks.

* **`degree`** = the plain count of neighbours (how many distinct characters this
  one shared a scene with).
* **`weighted_degree`** = the sum of the weights of all edges touching the node
  (the *total number of co-appearances*, counting repeats). Think of degree as
  "how many friends" and weighted degree as "how much total time spent with
  friends". A character can have few friends but spend a lot of screen time with
  them, which is why both numbers are useful.

We also store a `label` attribute equal to the character's name. Cytoscape, in
particular, likes an explicit `label`/`name` column to show on screen.

In [3]:
# Plain (unweighted) degree.
deg = dict(G.degree())
# Weighted degree: degree() can take weight='weight' to sum edge weights instead
# of just counting edges.
wdeg = dict(G.degree(weight="weight"))

nx.set_node_attributes(G, deg, "degree")
nx.set_node_attributes(G, wdeg, "weighted_degree")
nx.set_node_attributes(G, {n: n for n in G.nodes()}, "label")

# Sanity-check the top characters by each measure.
top_deg = sorted(deg.items(), key=lambda kv: -kv[1])[:8]
top_wdeg = sorted(wdeg.items(), key=lambda kv: -kv[1])[:8]
print("Top by degree (number of distinct co-stars):")
for name, d in top_deg:
    print(f"   {name:28s} degree={d:3d}  weighted_degree={wdeg[name]:4d}")
print("\nTop by weighted degree (total co-appearances):")
for name, w in top_wdeg:
    print(f"   {name:28s} weighted_degree={w:4d}  degree={deg[name]:3d}")


Top by degree (number of distinct co-stars):
   Harvey Dent / Two-Face       degree= 21  weighted_degree= 931
   Bruce Wayne / Batman         degree= 19  weighted_degree= 802
   Joker                        degree= 19  weighted_degree= 561
   Barbara Gordon               degree= 19  weighted_degree= 414
   Rachel Dawes                 degree= 16  weighted_degree= 496
   Mike Engel                   degree= 13  weighted_degree= 144
   Lau                          degree= 12  weighted_degree= 203
   Michael Wuertz               degree= 11  weighted_degree=  72

Top by weighted degree (total co-appearances):
   Harvey Dent / Two-Face       weighted_degree= 931  degree= 21
   Bruce Wayne / Batman         weighted_degree= 802  degree= 19
   Joker                        weighted_degree= 561  degree= 19
   Rachel Dawes                 weighted_degree= 496  degree= 16
   Barbara Gordon               weighted_degree= 414  degree= 19
   Lau                          weighted_degree= 203  degree= 

### Step 2 — Detect communities (for node colour)

A *community* is a clump of nodes that are more tightly connected to each other
than to the rest of the graph — in a movie, a sub-group of characters who mostly
share scenes among themselves (for example "the villains" versus "Gotham's
officials"). We colour nodes by community so the picture tells a story rather than
being a uniform blob.

We use the **Louvain method**, a fast and very popular community-detection
algorithm. In one sentence: it repeatedly merges nodes into groups in whatever way
most increases *modularity* — a score that is high when there are many edges inside
groups and few edges between groups. We pass `weight="weight"` so that strong
co-appearance links count more, and `seed=SEED` so the grouping is reproducible.

In [4]:
from networkx.algorithms.community import louvain_communities

communities = louvain_communities(G, weight="weight", seed=SEED)
# Map each node to an integer community id (0, 1, 2, ...).
comm_of = {}
for cid, members in enumerate(communities):
    for n in members:
        comm_of[n] = cid
nx.set_node_attributes(G, comm_of, "community")

print("Number of communities found:", len(communities))
for cid, members in enumerate(communities):
    sample = sorted(members, key=lambda n: -deg[n])[:4]
    print(f"  community {cid} ({len(members)} members), e.g.: {sample}")


Number of communities found: 3
  community 0 (13 members), e.g.: ['Harvey Dent / Two-Face', 'Bruce Wayne / Batman', 'Barbara Gordon', 'Rachel Dawes']
  community 1 (9 members), e.g.: ['Joker', 'Mike Engel', 'Chechen', 'Brian']
  community 2 (3 members), e.g.: ['Michael Wuertz', 'Judge Janet Surrillo', 'Anna Ramirez']


### Step 3 — Export to Gephi and Cytoscape file formats

We now write the network to the formats each program imports. Every file keeps the
edge `weight` and the node attributes (`degree`, `weighted_degree`, `community`,
`label`), so you can map size/colour onto them inside the GUI.

* **GEXF** (`.gexf`) — *Graph Exchange XML Format*, Gephi's native format.
* **GraphML** (`.graphml`) — a widely supported XML graph format; Cytoscape
  imports it directly.
* **Cytoscape.js JSON** (`.cyjs`) — the JSON shape Cytoscape uses internally. We
  produce it with `nx.cytoscape_data`, which returns a Python dictionary, and then
  `json.dump` it to disk.

In [5]:
gexf_path = os.path.join(EXPORT_DIR, "dark_knight.gexf")
graphml_path = os.path.join(EXPORT_DIR, "dark_knight.graphml")
cyjs_path = os.path.join(EXPORT_DIR, "dark_knight.cyjs")

# Gephi
nx.write_gexf(G, gexf_path)

# Cytoscape (GraphML)
nx.write_graphml(G, graphml_path)

# Cytoscape.js JSON
cyjs_data = nx.cytoscape_data(G)
with open(cyjs_path, "w") as fh:
    json.dump(cyjs_data, fh, indent=2)

for p in (gexf_path, graphml_path, cyjs_path):
    print(f"wrote {p}  ({os.path.getsize(p):,} bytes)")


wrote /home/mickaelz/Network analysis/exports/dark_knight.gexf  (33,610 bytes)
wrote /home/mickaelz/Network analysis/exports/dark_knight.graphml  (23,053 bytes)
wrote /home/mickaelz/Network analysis/exports/dark_knight.cyjs  (28,963 bytes)


### Step 4 — Emulated **Gephi** render (ForceAtlas2-style)

Gephi's signature look comes from its **ForceAtlas2** layout. ForceAtlas2 is a
*force-directed* layout: imagine every node is a little magnet that pushes all
other nodes away, while every edge is a spring that pulls its two endpoints
together. The drawing settles into a balance where tightly connected groups bunch
up and loosely connected nodes drift to the edges — exactly the readable,
"organic" look Gephi is known for.

NetworkX does not ship ForceAtlas2 by name, but its **`spring_layout`** (the
Fruchterman–Reingold algorithm) uses the same push/pull idea, so with tuned
settings it produces a very ForceAtlas2-like picture. The key knob is `k`, the
ideal distance between nodes: a larger `k` spreads the graph out more. We also run
many `iterations` so the springs fully settle.

In this render: **node size grows with degree**, **node colour encodes community**,
**edge thickness grows with co-appearance weight**, and we **label only the
high-degree characters** so the busiest nodes are named without cluttering the
picture.

In [6]:
# ForceAtlas2-like layout via Fruchterman-Reingold spring layout.
# Larger k spreads dense hubs apart so labels do not pile up in the centre.
pos_gephi = nx.spring_layout(G, k=2.2, iterations=600, weight="weight", seed=SEED)

degrees = np.array([deg[n] for n in G.nodes()])
# Node size scales with degree. The +base keeps even degree-1 nodes visible.
node_sizes = 90 + 75 * degrees
comm_ids = np.array([comm_of[n] for n in G.nodes()])

# A categorical colour map for communities (tab10 = 10 distinct colours).
cmap = plt.get_cmap("tab10")
node_colors = [cmap(c % 10) for c in comm_ids]

# Edge widths scale with weight, normalised so the thickest edge is ~5pt.
weights = np.array([d["weight"] for *_e, d in G.edges(data=True)])
edge_widths = 0.4 + 4.6 * (weights / weights.max())

fig, ax = plt.subplots(figsize=(14, 11))
nx.draw_networkx_edges(G, pos_gephi, ax=ax, width=edge_widths,
                       edge_color="#888888", alpha=0.40)
nx.draw_networkx_nodes(G, pos_gephi, ax=ax, node_size=node_sizes,
                       node_color=node_colors, edgecolors="white", linewidths=0.8)

# Label only the busiest characters (top 8 by degree) to avoid clutter. We nudge
# each label slightly above its node and give it a white background box so it
# stays readable even where nodes sit close together.
top_nodes = [n for n, _ in sorted(deg.items(), key=lambda kv: -kv[1])[:8]]
for n in top_nodes:
    x, y = pos_gephi[n]
    short = n.split(" / ")[0]   # short side of "Bruce Wayne / Batman"
    ax.text(x, y + 0.045, short, ha="center", va="bottom", fontsize=10,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))

# Community colour legend.
legend_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=cmap(c % 10),
           markeredgecolor="white", markersize=10, label=f"community {c}")
    for c in sorted(set(comm_ids))
]
ax.legend(handles=legend_handles, title="Louvain community",
          loc="upper left", framealpha=0.9)

ax.set_title("A4 (emulated Gephi) — The Dark Knight co-appearance network\n"
             "ForceAtlas2-style layout · node size = degree · colour = community · "
             "edge width = co-appearances", fontsize=12)
ax.axis("off")
ax.grid(False)
fig.tight_layout()
p = na.save_fig(fig, "partA4_emulated_gephi.png")
plt.close(fig)
print("saved", p)


saved /home/mickaelz/Network analysis/figures/partA4_emulated_gephi.png


### Step 5 — Emulated **Cytoscape** render (Kamada–Kawai layout, different style)

To make the two renders look genuinely different (as the task asks), the Cytoscape
emulation uses a **different layout and a different colour scheme**.

The layout here is **Kamada–Kawai**, another force-directed method but one that
works differently from spring layout: instead of simulating springs step by step,
it tries to make the *straight-line distance* between any two nodes on the page
match their *graph distance* (the number of hops along edges between them). The
result tends to be more symmetric and evenly spaced — closer to the tidy
"prefuse force-directed" look you get from Cytoscape's default layouts.

Here node colour is a smooth gradient by **degree** (light = peripheral character,
dark = central character) using the `viridis` colour map, which is a different
visual language from the categorical community colours above. Node size still grows
with degree, satisfying the core requirement.

In [7]:
# Kamada-Kawai layout (distance-matching force-directed layout).
pos_cyto = nx.kamada_kawai_layout(G, weight="weight")

# Colour by degree on a continuous viridis scale (distinct from the community map).
norm = plt.Normalize(vmin=degrees.min(), vmax=degrees.max())
cmap2 = plt.get_cmap("viridis")
node_colors2 = [cmap2(norm(deg[n])) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(13, 10))
nx.draw_networkx_edges(G, pos_cyto, ax=ax, width=0.6 + 3.0 * (weights / weights.max()),
                       edge_color="#b0b0b0", alpha=0.5)
nodes_drawn = nx.draw_networkx_nodes(
    G, pos_cyto, ax=ax, node_size=node_sizes, node_color=node_colors2,
    edgecolors="#333333", linewidths=0.6)

# Label the top characters again so the picture is interpretable.
top_nodes = [n for n, _ in sorted(deg.items(), key=lambda kv: -kv[1])[:8]]
labels = {n: n.split(" / ")[0] for n in top_nodes}
nx.draw_networkx_labels(G, pos_cyto, labels=labels, ax=ax, font_size=10,
                        font_weight="bold", font_color="#111111")

# Colourbar acting as the degree legend.
sm = plt.cm.ScalarMappable(cmap=cmap2, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("degree (number of co-stars)")

ax.set_title("A4 (emulated Cytoscape) — The Dark Knight co-appearance network\n"
             "Kamada-Kawai layout · node size = degree · colour = degree (viridis)",
             fontsize=12)
ax.axis("off")
ax.grid(False)
fig.tight_layout()
p = na.save_fig(fig, "partA4_emulated_cytoscape.png")
plt.close(fig)
print("saved", p)


saved /home/mickaelz/Network analysis/figures/partA4_emulated_cytoscape.png


### How I solved this task (A4)

**What I did.** I loaded the selected Dark Knight character co-appearance network
(25 characters, 106 co-appearance edges), enriched every node with two size-ready
attributes (`degree` and `weighted_degree`) and a community label, and then exported
the graph to the three import formats Gephi and Cytoscape understand
(`dark_knight.gexf`, `dark_knight.graphml`, `dark_knight.cyjs`). Finally I drew two
**emulated** renders that mimic the look of each program.

**Which method / tool.** For community colouring I used the **Louvain** algorithm
(maximises modularity to find tightly-knit groups). For the Gephi emulation I used
NetworkX `spring_layout` (Fruchterman–Reingold), which uses the same attract/repel
physics as Gephi's ForceAtlas2. For the Cytoscape emulation I used the
**Kamada–Kawai** layout with a continuous viridis degree-colouring, so the two
pictures look clearly different.

**Why I selected these.** This is a *headless* cluster node: there is no screen, so
the actual Gephi and Cytoscape windows cannot be opened here, and no real screenshot
can be captured by the agent. The honest, reproducible solution is to (1) hand you
the exact files those programs import, with the size/colour columns already baked in,
and (2) show what the finished visualization should look like via faithful matplotlib
emulations. Spring layout and Kamada–Kawai are the closest NetworkX equivalents of
the force-directed layouts those GUIs use.

**What the result means.** The picture shows a small, dense network with a few hub
characters. Batman, the Joker, and Harvey Dent / Two-Face are the largest nodes
because they share scenes with the most other characters — exactly the protagonist /
antagonist structure you would expect from the film. The community colours separate
loosely-connected clusters (for example minor characters who only appear with one or
two leads). To get the *official* GUI screenshots the assignment asks for, follow the
step-by-step import instructions in the writeup (`report/parts/partA_viz.md`).


# Task A7 — Lord of the Rings *couples* network

The goal here is to draw the network of **canonical romantic couples** in Tolkien's
legendarium (Lord of the Rings + The Silmarillion), with **vertex colour = gender**
and **vertex shape = race**.

**Data situation, explained honestly.** I searched for a ready-made LOTR dataset
that contains *both* gender and race *per character* plus a couples edge list. The
cleanest public source I found is **juandes/lotr-names-classification**
(`characters_data.csv`, 827 characters with a `race` column — races: *Ainur, Dwarf,
Elf, Hobbit, Man*). It has **no gender column and no couples list**, and several
female characters (Éowyn, Rosie, Goldberry, Lúthien, Lothíriel) are missing entirely.
So, exactly as the assignment anticipates, I **build the couples network by hand**
from canonical Tolkien pairings, attach gender and race to each character, and
**cross-check every race label against the public dataset** wherever the character
exists in it. The few labels not in that dataset come from canon (Tolkien's texts /
Tolkien Gateway) and are documented in the writeup.

In [8]:
# Load the public race reference we downloaded to data/lotr/.
import pandas as pd
race_ref_path = os.path.join(BASE, "data", "lotr", "characters_data.csv")
race_ref = pd.read_csv(race_ref_path)
race_lookup = dict(zip(race_ref["name"], race_ref["race"]))
print("Public race reference loaded:", race_ref.shape[0], "characters")
print("Races present:", sorted(race_ref["race"].unique()))


Public race reference loaded: 827 characters
Races present: ['Ainur', 'Dwarf', 'Elf', 'Hobbit', 'Man']


### Step 1 — Define the couples and the per-character attributes

Each entry below is a canonical pairing. For every person I record:

* **gender** — `male` / `female` (from canon; gender is binary for all these
  characters in Tolkien's texts).
* **race** — the character's people. I keep the public dataset's five-race scheme
  (*Man, Elf, Hobbit, Dwarf*) but **split "Ainur" into the finer canon terms**:
  *Maia* (a lesser divine spirit) for Melian and Goldberry, since lumping them with
  the great Valar would hide a meaningful distinction. **Tom Bombadil** is famously
  of *unknown* race in canon ("Eldest, that's what I am"), so I label him `Unknown`
  rather than guess. These choices are spelled out in the writeup.

The `from_public` flag records whether the race label is confirmed by the public
dataset (a small reproducibility nicety).

In [9]:
# Canonical LOTR / Silmarillion couples. Each tuple: (person A, person B).
COUPLES = [
    ("Aragorn", "Arwen"),
    ("Samwise Gamgee", "Rosie Cotton"),
    ("Faramir", "Eowyn"),
    ("Beren", "Luthien"),
    ("Galadriel", "Celeborn"),
    ("Tom Bombadil", "Goldberry"),
    ("Eomer", "Lothiriel"),
    ("Thingol", "Melian"),
    ("Tuor", "Idril"),
]

# Per-character attributes (gender + race). Race cross-checked vs the public
# dataset where the character exists there; finer/extra labels from canon.
CHAR_ATTRS = {
    "Aragorn":        {"gender": "male",   "race": "Man"},     # public: Aragorn II = Man
    "Arwen":          {"gender": "female", "race": "Elf"},     # public: Elf
    "Samwise Gamgee": {"gender": "male",   "race": "Hobbit"},  # public: Hobbit
    "Rosie Cotton":   {"gender": "female", "race": "Hobbit"},  # canon (not in public set)
    "Faramir":        {"gender": "male",   "race": "Man"},     # public: Man
    "Eowyn":          {"gender": "female", "race": "Man"},     # canon (Rohan); not in public set
    "Beren":          {"gender": "male",   "race": "Man"},     # public: Man
    "Luthien":        {"gender": "female", "race": "Elf"},     # canon (half-Maia/Elf); not in public set
    "Galadriel":      {"gender": "female", "race": "Elf"},     # public: Elf
    "Celeborn":       {"gender": "male",   "race": "Elf"},     # public: Elf
    "Tom Bombadil":   {"gender": "male",   "race": "Unknown"}, # canon: race deliberately unknown
    "Goldberry":      {"gender": "female", "race": "Maia"},    # canon: River-daughter / Maia-like spirit
    "Eomer":          {"gender": "male",   "race": "Man"},     # public: Man
    "Lothiriel":      {"gender": "female", "race": "Man"},     # canon (Dol Amroth); not in public set
    "Thingol":        {"gender": "male",   "race": "Elf"},     # public: Elf
    "Melian":         {"gender": "female", "race": "Maia"},    # public lumps as 'Ainur'; canon = Maia
    "Tuor":           {"gender": "male",   "race": "Man"},     # public: Man
    "Idril":          {"gender": "female", "race": "Elf"},     # public: Elf
}

# Annotate each character with whether the race matches the public reference.
for name, attrs in CHAR_ATTRS.items():
    pub = race_lookup.get(name) or race_lookup.get(name + " II")
    attrs["from_public"] = bool(pub) and (
        pub == attrs["race"] or (pub == "Ainur" and attrs["race"] == "Maia"))

confirmed = sum(a["from_public"] for a in CHAR_ATTRS.values())
print(f"{confirmed} of {len(CHAR_ATTRS)} race labels confirmed by the public dataset.")
print("(The rest are female/unknown characters absent from that dataset; "
      "labels come from Tolkien canon.)")


12 of 18 race labels confirmed by the public dataset.
(The rest are female/unknown characters absent from that dataset; labels come from Tolkien canon.)


### Step 2 — Build the graph and attach attributes

We create an undirected graph, add one edge per couple (a couple is a relationship
between two people, and relationships have no direction), and attach `gender` and
`race` to every node.

In [10]:
L = nx.Graph()
L.add_edges_from(COUPLES)
for name, attrs in CHAR_ATTRS.items():
    L.nodes[name]["gender"] = attrs["gender"]
    L.nodes[name]["race"] = attrs["race"]
    L.nodes[name]["label"] = name

print("LOTR couples network:")
print("  nodes (characters):", L.number_of_nodes())
print("  edges (couples):   ", L.number_of_edges())
print("\nGender breakdown:")
import collections
print("  ", dict(collections.Counter(nx.get_node_attributes(L, "gender").values())))
print("Race breakdown:")
print("  ", dict(collections.Counter(nx.get_node_attributes(L, "race").values())))


LOTR couples network:
  nodes (characters): 18
  edges (couples):    9

Gender breakdown:
   {'male': 9, 'female': 9}
Race breakdown:
   {'Man': 7, 'Elf': 6, 'Hobbit': 2, 'Unknown': 1, 'Maia': 2}


### Step 3 — Export to Gephi and Cytoscape formats (with gender + race)

Same three formats as A4, but now the load-bearing node attributes are **`gender`**
and **`race`**, because in Cytoscape you will map *fill colour → gender* and
*shape → race*.

In [11]:
lotr_gexf = os.path.join(EXPORT_DIR, "lotr_couples.gexf")
lotr_graphml = os.path.join(EXPORT_DIR, "lotr_couples.graphml")
lotr_cyjs = os.path.join(EXPORT_DIR, "lotr_couples.cyjs")

nx.write_gexf(L, lotr_gexf)
nx.write_graphml(L, lotr_graphml)
with open(lotr_cyjs, "w") as fh:
    json.dump(nx.cytoscape_data(L), fh, indent=2)

for p in (lotr_gexf, lotr_graphml, lotr_cyjs):
    print(f"wrote {p}  ({os.path.getsize(p):,} bytes)")


wrote /home/mickaelz/Network analysis/exports/lotr_couples.gexf  (4,504 bytes)
wrote /home/mickaelz/Network analysis/exports/lotr_couples.graphml  (3,044 bytes)
wrote /home/mickaelz/Network analysis/exports/lotr_couples.cyjs  (4,990 bytes)


### Step 4 — Emulated render: colour = gender, shape = race

matplotlib cannot draw different marker *shapes* in a single `draw_networkx_nodes`
call (one call uses one shape). So the trick is to **draw the nodes race-by-race**:
for each race we issue a separate scatter call with that race's marker shape, while
the *colour* of each dot is decided by the character's gender. That gives us the two
independent visual channels the task wants — **shape tells you the race, colour tells
you the gender** — and lets us build a clean two-part legend.

In [12]:
# Reproducible layout for the couples network.
pos_lotr = nx.spring_layout(L, k=1.4, iterations=400, seed=SEED)

# Visual encodings.
GENDER_COLOR = {"male": "#1f77b4", "female": "#d62728"}   # blue / red
RACE_MARKER = {
    "Man":     "o",   # circle
    "Elf":     "^",   # triangle
    "Hobbit":  "s",   # square
    "Maia":    "*",   # star
    "Dwarf":   "D",   # diamond (no dwarf couple here, kept for completeness)
    "Unknown": "X",   # cross
}

fig, ax = plt.subplots(figsize=(12, 9))

# Edges first, so nodes sit on top.
nx.draw_networkx_edges(L, pos_lotr, ax=ax, edge_color="#999999", width=1.8, alpha=0.7)

# Draw nodes one race at a time (each race = its own marker shape).
races_present = sorted(set(nx.get_node_attributes(L, "race").values()))
for race in races_present:
    nodes_r = [n for n in L.nodes() if L.nodes[n]["race"] == race]
    xs = [pos_lotr[n][0] for n in nodes_r]
    ys = [pos_lotr[n][1] for n in nodes_r]
    colors = [GENDER_COLOR[L.nodes[n]["gender"]] for n in nodes_r]
    ax.scatter(xs, ys, s=520, c=colors, marker=RACE_MARKER[race],
               edgecolors="white", linewidths=1.4, zorder=3)

# Character labels (small network, so we can label everyone).
for n in L.nodes():
    x, y = pos_lotr[n]
    ax.text(x, y - 0.085, n, ha="center", va="top", fontsize=8.5)

# Two-part legend: one for gender (colour), one for race (shape).
gender_handles = [
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=c,
           markeredgecolor="white", markersize=12, label=g)
    for g, c in GENDER_COLOR.items()
]
race_handles = [
    Line2D([0], [0], marker=RACE_MARKER[r], linestyle="none",
           markerfacecolor="#555555", markeredgecolor="white", markersize=12, label=r)
    for r in races_present
]
leg1 = ax.legend(handles=gender_handles, title="Gender (colour)",
                 loc="upper left", framealpha=0.9)
ax.add_artist(leg1)   # keep first legend when adding the second
ax.legend(handles=race_handles, title="Race (shape)",
          loc="lower left", framealpha=0.9)

ax.set_title("A7 — Lord of the Rings couples network\n"
             "colour = gender · shape = race", fontsize=13)
ax.axis("off")
ax.grid(False)
fig.tight_layout()
p = na.save_fig(fig, "partA7_lotr_couples.png")
plt.close(fig)
print("saved", p)


saved /home/mickaelz/Network analysis/figures/partA7_lotr_couples.png


### How I solved this task (A7)

**What I did.** I assembled a Lord of the Rings *couples* network: nine canonical
romantic pairings (Aragorn–Arwen, Samwise–Rosie, Faramir–Éowyn, Beren–Lúthien,
Galadriel–Celeborn, Tom Bombadil–Goldberry, Éomer–Lothíriel, Thingol–Melian,
Tuor–Idril) spanning 18 characters. To every character I attached a **gender** and a
**race**, then exported the network to Gephi/Cytoscape formats and drew an emulated
render where **colour shows gender and marker shape shows race**.

**Which method / tool / data.** I started from the public **juandes**
`characters_data.csv` race dataset (827 characters; races *Ainur, Dwarf, Elf, Hobbit,
Man*), which I downloaded to `data/lotr/`. Because that file has no gender column, no
couples list, and is missing several of the female partners, I built the edge list by
hand from Tolkien canon and confirmed each race label against the public dataset where
possible (most of the male partners and the named Elf women are confirmed). For the
emulated drawing I used NetworkX `spring_layout` and matplotlib, drawing nodes
**one race at a time** so each race can have its own marker shape.

**Why these design choices.** Both gender and race are *categorical* attributes (named
buckets, not numbers). **Colour is the most eye-catching visual channel**, so I used it
for the **binary** gender split (just two values → easy to read as two colours).
**Shape** is great at separating a *small number* of categories, so I used it for the
handful of races. This keeps the two channels independent and instantly readable: a red
triangle is a female Elf, a blue circle is a male Man, and so on.

**What the result means.** The picture is a set of disconnected two-person components
(the couples don't link to each other), which is correct: it visualizes *who is paired
with whom*, not a connected social web. The colour/shape encoding immediately reveals
recurring patterns in the legendarium, most strikingly the famous **Man + Elf** unions
(Aragorn–Arwen, Beren–Lúthien, Tuor–Idril) — mortal-meets-immortal love stories that
are a central theme of Tolkien's mythology. For the *official* Cytoscape screenshot,
follow the discrete-mapping instructions in the writeup.
